# TFM – Modelado YOLO · Expresiones Faciales

**Detección y clasificación de expresiones faciales con YOLOv8**
Dataset: `YOLO_EXPRESIONES_UNIFIED` · 7 clases (`angry, disgust, fear, happy, neutral, sad, surprise`)

Este notebook **da continuidad al análisis exploratorio** (`TFM_EDA_YOLO_Expresiones.ipynb`).
El EDA ya dejó preparado:

- La estructura `images/{train,val,test}` y `labels/{train,val,test}`.
- El archivo `data.yaml` con las 7 clases.
- El **equilibrado del `train`** mediante aumentación (Albumentations) de las clases minoritarias.

Aquí se entrena el modelo base, se evalúa en `test`, se extraen las métricas reales
y se generan las figuras para la documentación y la sustentación.

> **Antes de ejecutar:** `Runtime → Change runtime type → GPU (T4 / A100) + High-RAM`.


## 🖥️ Sección 1 – Montar Drive y verificar GPU / RAM

In [ ]:
# ============================================================
# CELDA 1 – Montar Drive y verificar GPU / RAM
# ============================================================
import os, sys, subprocess, shutil
import torch`



`

# --- Montar Drive (limpiando si quedó un montaje sucio) ---
mountpoint = "/content/drive"
try:
    from google.colab import drive
    if os.path.ismount(mountpoint):
        print("ℹ️  Google Drive ya estaba montado, continuando...")
    else:
        if os.path.isdir(mountpoint) and os.listdir(mountpoint):
            shutil.rmtree(mountpoint)
            os.makedirs(mountpoint)
            print("🧹 Directorio de montaje limpiado.")
        drive.mount(mountpoint, force_remount=True)
        print("✅ Google Drive montado correctamente.")
except Exception as e:
    print(f"ℹ️  No se montó Drive automáticamente ({e}). Si ya está montado, no hay problema.")

# --- Verificación de hardware ---
print("=" * 70)
print("🖥️  VERIFICACIÓN DE HARDWARE")
print("=" * 70)
print(f"🐍 Python : {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print("✅ CUDA disponible")
    print(f"🎯 GPU    : {torch.cuda.get_device_name(0)}")
    print(f"🧩 CUDA   : {torch.version.cuda}")
else:
    print("⚠️  CUDA no disponible. Activa GPU en Runtime → Change runtime type → GPU.")

# --- RAM del sistema ---
try:
    with open("/proc/meminfo") as fh:
        lines = fh.read().split("\n")
    mem_total_kb = int([l for l in lines if l.startswith("MemTotal")][0].split()[1])
    mem_avail_kb = int([l for l in lines if l.startswith("MemAvailable")][0].split()[1])
    print(f"💾 RAM    : {mem_total_kb/1024**2:.1f} GB totales | {mem_avail_kb/1024**2:.1f} GB disponibles")
    if mem_total_kb / 1024**2 >= 50:
        print("✅ High-RAM activo (≥50 GB)")
    else:
        print("⚠️  RAM estándar (~12 GB). Para High-RAM: Runtime → Change runtime type.")
except Exception as e:
    print(f"⚠️  No se pudo leer la RAM: {e}")


Mounted at /content/drive
✅ Google Drive montado correctamente.
🖥️  VERIFICACIÓN DE HARDWARE
🐍 Python : 3.12.13
🔥 PyTorch: 2.11.0+cu128
✅ CUDA disponible
🎯 GPU    : Tesla T4
🧩 CUDA   : 12.8
💾 RAM    : 51.0 GB totales | 49.0 GB disponibles
✅ High-RAM activo (≥50 GB)


## 📦 Sección 2 – Instalar Ultralytics (YOLO)

In [2]:
# ============================================================
# CELDA 2 – Instalar Ultralytics YOLO
# ============================================================
!pip install -q ultralytics

import ultralytics
from ultralytics import YOLO

ultralytics.checks()


Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (8 CPUs, 51.0 GB RAM, 47.1/235.7 GB disk)


## 📁 Sección 3 – Rutas del dataset de expresiones y `data.yaml`

Se reutiliza exactamente el dataset que dejó preparado el EDA. Si por algún motivo
faltara `data.yaml`, esta celda lo recrea con las 7 clases en el orden esperado.

In [3]:
# ============================================================
# CELDA 3 – Rutas del dataset de EXPRESIONES y data.yaml
# ============================================================
from pathlib import Path
import yaml

# --- Rutas base (mismas del EDA) ---
TFM_DIR      = Path("/content/drive/MyDrive/TFM")
BASE         = TFM_DIR / "EXPRESIONES"
YOLO_DATASET = BASE / "YOLO_EXPRESIONES_UNIFIED"
DATA_YAML    = YOLO_DATASET / "data.yaml"

SPLITS   = ["train", "val", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# --- Orden de clases (= class_id en los .txt). Debe coincidir con el EDA ---
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = len(CLASS_NAMES)

print("=" * 70)
print("📁 DATASET PARA ENTRENAMIENTO")
print("=" * 70)
print(f"Dataset : {YOLO_DATASET}")
print(f"YAML    : {DATA_YAML}")
print(f"Existe dataset?: {YOLO_DATASET.exists()}")
print(f"Existe YAML?   : {DATA_YAML.exists()}")

# --- Crear data.yaml si no existe (red de seguridad) ---
if not DATA_YAML.exists():
    YOLO_DATASET.mkdir(parents=True, exist_ok=True)
    data_yaml_content = {
        "path":  str(YOLO_DATASET),
        "train": "images/train",
        "val":   "images/val",
        "test":  "images/test",
        "nc":    NUM_CLASSES,
        "names": CLASS_NAMES,
    }
    with open(DATA_YAML, "w", encoding="utf-8") as f:
        yaml.dump(data_yaml_content, f, default_flow_style=False,
                  allow_unicode=True, sort_keys=False)
    print("⚠️  data.yaml no existía → se creó automáticamente.")

# --- Leer y mostrar data.yaml ---
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

print("\n📄 Contenido data.yaml:")
for k, v in data_yaml.items():
    print(f"   {k}: {v}")

# Alinear CLASS_NAMES con el yaml por si difieren
yaml_names = data_yaml.get("names")
if yaml_names and list(yaml_names) != CLASS_NAMES:
    print("\n⚠️  Las clases del yaml difieren de las del notebook. Se usan las del yaml.")
    CLASS_NAMES = list(yaml_names)
    NUM_CLASSES = len(CLASS_NAMES)

print("\n🎭 Clases:")
for i, c in enumerate(CLASS_NAMES):
    print(f"   {i}: {c}")

print("\n📂 Verificación de carpetas:")
for split in SPLITS:
    img_ok = (YOLO_DATASET / "images" / split).exists()
    lbl_ok = (YOLO_DATASET / "labels" / split).exists()
    print(f"   {split:<6} → images: {'✅' if img_ok else '❌'}  labels: {'✅' if lbl_ok else '❌'}")


📁 DATASET PARA ENTRENAMIENTO
Dataset : /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED
YAML    : /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/data.yaml
Existe dataset?: True
Existe YAML?   : True

📄 Contenido data.yaml:
   path: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED
   train: images/train
   val: images/val
   test: images/test
   nc: 7
   names: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

🎭 Clases:
   0: angry
   1: disgust
   2: fear
   3: happy
   4: neutral
   5: sad
   6: surprise

📂 Verificación de carpetas:
   train  → images: ✅  labels: ✅
   val    → images: ✅  labels: ✅
   test   → images: ✅  labels: ✅


## 🚆 Sección 4 – Entrenamiento del modelo base

Se entrena `yolov8n.pt` como **línea base**, igual que en el trabajo de caídas, para que
los resultados sean comparables. A diferencia de aquel (binario), aquí el problema es
**multiclase (7 expresiones)**, así que se dejan algo más de épocas y `patience` para que
el modelo aprenda las clases más sutiles.

> El desbalanceo entre expresiones **ya fue mitigado en el EDA** mediante aumentación de las
> clases minoritarias del `train`. Si quisieras más capacidad puedes cambiar `MODEL_NAME` a
> `yolov8s.pt`.

In [4]:
# ============================================================
# CELDA 4 – Entrenamiento base YOLO (multiclase – 7 expresiones)
# ============================================================
from ultralytics import YOLO

# yolov8n.pt = liviano y rápido (línea base, comparable con caídas)
# yolov8s.pt = mayor capacidad, algo más lento
MODEL_NAME = "yolov8n.pt"

PROJECT_DIR = str(TFM_DIR / "runs_yolo_expresiones")
RUN_NAME    = "baseline_yolov8n_expresiones"

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=80,          # 7 clases sutiles → algo más que en el binario de caídas
    imgsz=640,          # mismo tamaño que el baseline de caídas (comparable)
    batch=16,
    patience=20,        # early stopping
    seed=42,            # reproducibilidad
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    workers=2,
    plots=True,
)

print("\n✅ Entrenamiento finalizado.")
print(f"📁 Resultados en: {PROJECT_DIR}/{RUN_NAME}")


Streaming output truncated to the last 5000 lines.
train: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019167.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019167.jpg'
train: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019179.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019179.jpg'
train: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019181.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019181.jpg'
train: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train/AFFECTNET_._image0019182.jpg

### 🔎 Sección 4b – Comprobar imágenes y labels en `train` / `val`

In [5]:
# ============================================================
# CELDA 4B – Comprobar cantidad de imágenes y labels en train y val
# ============================================================
import pandas as pd

filas = []
for split in ["train", "val"]:
    img_dir = YOLO_DATASET / "images" / split
    lbl_dir = YOLO_DATASET / "labels" / split

    imgs = sorted([p for p in img_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in IMG_EXTS]) if img_dir.exists() else []
    lbls = sorted([p for p in lbl_dir.iterdir()
                   if p.is_file() and p.suffix.lower() == ".txt"]) if lbl_dir.exists() else []

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    filas.append({
        "split": split,
        "existe_images": img_dir.exists(),
        "existe_labels": lbl_dir.exists(),
        "cantidad_imagenes": len(imgs),
        "cantidad_labels": len(lbls),
        "imagenes_sin_label": len(img_stems - lbl_stems),
        "labels_sin_imagen": len(lbl_stems - img_stems),
    })

df_train_val = pd.DataFrame(filas)
print("=" * 70)
print("📋 COMPROBACIÓN DE TRAIN Y VAL")
print("=" * 70)
display(df_train_val)


📋 COMPROBACIÓN DE TRAIN Y VAL


,split,existe_images,existe_labels,cantidad_imagenes,cantidad_labels,imagenes_sin_label,labels_sin_imagen
0,train,True,True,19842,19842,0,0
1,val,True,True,4685,4685,0,0


## 🧭 Sección 5 – Rutas para la evaluación del modelo entrenado

In [6]:
# ============================================================
# CELDA 5 – Definir rutas para evaluación del modelo entrenado
# ============================================================
PROJECT_DIR = TFM_DIR / "runs_yolo_expresiones"
RUN_NAME    = "baseline_yolov8n_expresiones"

BEST_MODEL = PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
LAST_MODEL = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"

print("=" * 70)
print("📁 RUTAS PARA EVALUACIÓN")
print("=" * 70)
print(f"📁 Dataset    : {YOLO_DATASET}")
print(f"📄 data.yaml  : {DATA_YAML}")
print(f"🏆 best.pt    : {BEST_MODEL}")
print(f"📦 last.pt    : {LAST_MODEL}")

print("\nVerificación:")
print(f"Existe dataset?: {YOLO_DATASET.exists()}")
print(f"Existe YAML?   : {DATA_YAML.exists()}")
print(f"Existe best.pt?: {BEST_MODEL.exists()}")
print(f"Existe last.pt?: {LAST_MODEL.exists()}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"No se encontró DATA_YAML:\n{DATA_YAML}")
if not BEST_MODEL.exists():
    raise FileNotFoundError(f"No se encontró BEST_MODEL (¿entrenaste la celda 4?):\n{BEST_MODEL}")


📁 RUTAS PARA EVALUACIÓN
📁 Dataset    : /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED
📄 data.yaml  : /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/data.yaml
🏆 best.pt    : /content/drive/MyDrive/TFM/runs_yolo_expresiones/baseline_yolov8n_expresiones/weights/best.pt
📦 last.pt    : /content/drive/MyDrive/TFM/runs_yolo_expresiones/baseline_yolov8n_expresiones/weights/last.pt

Verificación:
Existe dataset?: True
Existe YAML?   : True
Existe best.pt?: True
Existe last.pt?: True


## 🧪 Sección 6 – Evaluación final en `TEST`

In [ ]:
# ============================================================
# CELDA 6 – Evaluación final en TEST
# ============================================================
from ultralytics import YOLO

model_best = YOLO(str(BEST_MODEL))

print("=" * 70)
print("🧪 EVALUACIÓN FINAL EN TEST")
print("=" * 70)
print(f"🏆 Modelo     : {BEST_MODEL}")
print(f"📄 data.yaml  : {DATA_YAML}")

metrics_test = model_best.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,                       # genera matriz de confusión y curvas PR/F1/P/R
    project=str(PROJECT_DIR),
    name=f"{RUN_NAME}_test_eval",
    exist_ok=True,
)

print("\n" + "=" * 70)
print("✅ EVALUACIÓN TEST FINALIZADA")
print("=" * 70)
print(f"Precision (global): {metrics_test.box.mp:.4f}")
print(f"Recall    (global): {metrics_test.box.mr:.4f}")
print(f"mAP50     (global): {metrics_test.box.map50:.4f}")
print(f"mAP50-95  (global): {metrics_test.box.map:.4f}")

RESULTS_DIR = PROJECT_DIR / f"{RUN_NAME}_test_eval"
print(f"\n📁 Resultados guardados en:\n{RESULTS_DIR}")


🧪 EVALUACIÓN FINAL EN TEST
🏆 Modelo     : /content/drive/MyDrive/TFM/runs_yolo_expresiones/baseline_yolov8n_expresiones/weights/best.pt
📄 data.yaml  : /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/data.yaml
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 0.0±0.0 MB/s, size: 4.4 KB)
val: Scanning /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/labels/test... 1866 images, 0 backgrounds, 2077 corrupt: 100% ━━━━━━━━━━━━ 3586/3586 35.0it/s 1:42
val: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/test/AFFECTNET_._image0000027.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/test/AFFECTNET_._image0000027.jpg'
val: /content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/test/A

## 📊 Sección 7 – Resumen de métricas `TEST` (extraídas del modelo)

A diferencia del notebook de caídas (donde las métricas se escribieron a mano),
aquí se **leen directamente del objeto `metrics_test`**, por lo que la tabla es
siempre fiel al modelo entrenado, clase por clase.

In [ ]:
# ============================================================
# CELDA 7 – Resumen de métricas TEST por clase (leídas del modelo)
# ============================================================
import numpy as np
import pandas as pd

RESULTS_DIR = PROJECT_DIR / f"{RUN_NAME}_test_eval"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def extraer_metricas_por_clase(metrics, class_names):
    """Construye un DataFrame con métricas global + por clase desde un objeto val()."""
    box = metrics.box
    filas = [{
        "clase": "Global (all)",
        "precision": float(box.mp),
        "recall":    float(box.mr),
        "mAP50":     float(box.map50),
        "mAP50_95":  float(box.map),
    }]
    # Arrays por clase, alineados a box.ap_class_index
    idx  = list(box.ap_class_index)
    p    = np.atleast_1d(box.p)
    r    = np.atleast_1d(box.r)
    ap50 = np.atleast_1d(box.ap50)
    ap   = np.atleast_1d(box.ap)
    for k, c in enumerate(idx):
        nombre = class_names[c] if c < len(class_names) else str(c)
        filas.append({
            "clase": nombre,
            "precision": float(p[k])    if k < len(p)    else np.nan,
            "recall":    float(r[k])    if k < len(r)    else np.nan,
            "mAP50":     float(ap50[k]) if k < len(ap50) else np.nan,
            "mAP50_95":  float(ap[k])   if k < len(ap)   else np.nan,
        })
    return pd.DataFrame(filas)

df_metricas_test = extraer_metricas_por_clase(metrics_test, CLASS_NAMES)

# Guardar CSV / TXT
OUT_CSV = RESULTS_DIR / "metricas_test_resumen.csv"
OUT_TXT = RESULTS_DIR / "metricas_test_resumen.txt"
df_metricas_test.to_csv(OUT_CSV, index=False)
with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("Resumen métricas TEST - YOLOv8n (Expresiones)\n")
    f.write("=" * 60 + "\n")
    f.write(df_metricas_test.round(4).to_string(index=False))
    f.write("\n")

print("✅ Resumen de métricas guardado en:")
print(f"   {OUT_CSV}")
print(f"   {OUT_TXT}")
print("\n📋 Métricas TEST por clase:")
display(df_metricas_test.round(4))


## 🖼️ Sección 8 – Inferencia visual sobre imágenes de `TEST`

In [ ]:
# ============================================================
# CELDA 8 – Inferencia visual sobre imágenes de TEST
# ============================================================
import random
from ultralytics import YOLO

model_best = YOLO(str(BEST_MODEL))

test_img_dir = YOLO_DATASET / "images" / "test"
imgs_test = [p for p in test_img_dir.iterdir()
             if p.is_file() and p.suffix.lower() in IMG_EXTS]
print(f"Imágenes TEST encontradas: {len(imgs_test):,}")

random.seed(42)
muestras = random.sample(imgs_test, min(30, len(imgs_test)))

PRED_PROJECT = TFM_DIR / "predicciones_yolo_expresiones"
PRED_NAME    = "test_visual_baseline_yolov8n"

pred_results = model_best.predict(
    source=[str(p) for p in muestras],
    imgsz=640,
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(PRED_PROJECT),
    name=PRED_NAME,
    exist_ok=True,
)

print("\n✅ Predicciones visuales guardadas en:")
print(PRED_PROJECT / PRED_NAME)


### 🎨 Sección 9 – Mostrar predicciones visuales guardadas

In [ ]:
# ============================================================
# CELDA 9 – Mostrar predicciones visuales guardadas
# ============================================================
import math, cv2
import matplotlib.pyplot as plt

PRED_DIR = TFM_DIR / "predicciones_yolo_expresiones" / "test_visual_baseline_yolov8n"
print(f"📁 Carpeta de predicciones: {PRED_DIR}  (existe: {PRED_DIR.exists()})")

pred_imgs = [p for p in PRED_DIR.iterdir()
             if p.is_file() and p.suffix.lower() in IMG_EXTS]
print(f"Predicciones encontradas: {len(pred_imgs):,}")
if len(pred_imgs) == 0:
    raise RuntimeError("No se encontraron imágenes de predicción para mostrar.")

random.seed(42)
muestras = random.sample(pred_imgs, min(12, len(pred_imgs)))
cols = 4
rows = math.ceil(len(muestras) / cols)

plt.figure(figsize=(16, 4 * rows))
for i, img_path in enumerate(muestras, start=1):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(img_path.name[:24], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()


### 🔎 Sección 10 – Comparación predicción vs etiqueta real

Para expresiones cada imagen tiene una expresión dominante, así que se compara la
**clase de mayor confianza** predicha contra la clase real (la más frecuente en el
label). Se obtiene un DataFrame de aciertos/errores y la exactitud sobre la muestra.

In [ ]:
# ============================================================
# CELDA 10 – Comparar predicción vs etiqueta real en muestras
# ============================================================
GT_LABEL_DIR   = YOLO_DATASET / "labels" / "test"
PRED_DIR       = TFM_DIR / "predicciones_yolo_expresiones" / "test_visual_baseline_yolov8n"
PRED_LABEL_DIR = PRED_DIR / "labels"

print("=" * 70)
print("🔎 COMPARACIÓN PREDICCIÓN VS ETIQUETA REAL")
print("=" * 70)
print(f"📁 GT labels   : {GT_LABEL_DIR}")
print(f"📁 Pred labels : {PRED_LABEL_DIR}")

def leer_gt_label(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    clases = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    clases.append(int(float(parts[0])))
                except Exception:
                    pass
    return max(set(clases), key=clases.count) if clases else None

def leer_pred_label(path):
    if not path.exists() or path.stat().st_size == 0:
        return None, None
    preds = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                try:
                    preds.append((int(float(parts[0])), float(parts[5])))
                except Exception:
                    pass
    if not preds:
        return None, None
    pred = max(preds, key=lambda x: x[1])
    return pred[0], pred[1]

def nombre_clase(cid):
    return CLASS_NAMES[cid] if cid is not None and 0 <= cid < len(CLASS_NAMES) else None

filas = []
if PRED_LABEL_DIR.exists():
    pred_files = sorted(p for p in PRED_LABEL_DIR.iterdir() if p.suffix.lower() == ".txt")
    for pred_path in pred_files:
        stem = pred_path.stem
        gt_cls = leer_gt_label(GT_LABEL_DIR / f"{stem}.txt")
        pred_cls, conf = leer_pred_label(pred_path)
        filas.append({
            "imagen": stem,
            "gt_class": nombre_clase(gt_cls) or "SIN_GT",
            "pred_class": nombre_clase(pred_cls) or "SIN_PRED",
            "confidence": conf,
            "correcto": (gt_cls is not None and gt_cls == pred_cls),
        })

df_comparacion = pd.DataFrame(filas)
print("\n📋 Comparación:")
display(df_comparacion)

if len(df_comparacion) > 0:
    acc = df_comparacion["correcto"].mean() * 100
    print(f"\n✅ Exactitud en estas muestras: {acc:.2f}%")
    errores = df_comparacion[~df_comparacion["correcto"]]
    if len(errores) > 0:
        print("\n⚠️ Errores encontrados:")
        display(errores)
    else:
        print("\n✅ Sin errores en las predicciones guardadas de la muestra.")


## 🗂️ Sección 11 – Copiar figuras oficiales de YOLO para documentación

In [ ]:
# ============================================================
# CELDA 11 – Copiar figuras oficiales de YOLO (matriz, curvas, batches)
# ============================================================
import shutil

TEST_EVAL_DIR = PROJECT_DIR / f"{RUN_NAME}_test_eval"
FIG_DIR = TFM_DIR / "figuras_resultados_yolo_expresiones"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Según la versión de Ultralytics, las curvas pueden venir con prefijo "Box".
candidatos = [
    "confusion_matrix.png", "confusion_matrix_normalized.png",
    "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png",
    "BoxPR_curve.png", "BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png",
    "val_batch0_labels.jpg", "val_batch0_pred.jpg",
    "val_batch1_labels.jpg", "val_batch1_pred.jpg",
    "val_batch2_labels.jpg", "val_batch2_pred.jpg",
    "metricas_test_resumen.csv", "metricas_test_resumen.txt",
]

print("=" * 70)
print("📁 COPIANDO FIGURAS OFICIALES DE TEST")
print("=" * 70)
print(f"Origen : {TEST_EVAL_DIR}")
print(f"Destino: {FIG_DIR}")

copiados, no_encontrados = [], []
for nombre in candidatos:
    src = TEST_EVAL_DIR / nombre
    if src.exists():
        # las figuras llevan prefijo test_ ; los csv/txt se copian tal cual
        destino = nombre if nombre.endswith((".csv", ".txt")) else f"test_{nombre}"
        dst = FIG_DIR / destino
        shutil.copy2(src, dst)
        copiados.append(dst.name)
        print(f"✅ Copiado: {dst.name}")
    else:
        no_encontrados.append(nombre)

print(f"\n✅ Copiados: {len(copiados)} | ⚠️ No encontrados: {len(no_encontrados)}")
print(f"📁 Figuras disponibles en: {FIG_DIR}")


## 📈 Sección 12 – Gráficas de métricas `TEST` por clase

In [ ]:
# ============================================================
# CELDA 12 – Gráficas de métricas TEST por clase (barras + heatmap)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = TFM_DIR / "figuras_resultados_yolo_expresiones"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Cargar la tabla de métricas (la generada en la celda 7)
CSV_METRICAS = (PROJECT_DIR / f"{RUN_NAME}_test_eval" / "metricas_test_resumen.csv")
if "df_metricas_test" in globals():
    df_m = df_metricas_test.copy()
elif CSV_METRICAS.exists():
    df_m = pd.read_csv(CSV_METRICAS)
else:
    raise RuntimeError("No hay métricas. Ejecuta primero las celdas 6 y 7.")

metricas = ["precision", "recall", "mAP50", "mAP50_95"]
labels_m = ["Precision", "Recall", "mAP50", "mAP50-95"]

# --- 1. Barras agrupadas ---
clases = df_m["clase"].tolist()
x = np.arange(len(clases))
width = 0.18

fig, ax = plt.subplots(figsize=(max(12, len(clases) * 1.4), 6.5))
for i, (m, lab) in enumerate(zip(metricas, labels_m)):
    pos = x + (i - 1.5) * width
    vals = df_m[m].values
    barras = ax.bar(pos, vals, width, label=lab)
    for b, v in zip(barras, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.2f}",
                ha="center", va="bottom", fontsize=8, rotation=90)

ax.set_title("Métricas de desempeño YOLOv8n en TEST — Expresiones",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Clase evaluada", fontsize=12)
ax.set_ylabel("Valor de la métrica", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(clases, rotation=20, ha="right", fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=True)
plt.tight_layout()

OUT_BARRAS_PNG = FIG_DIR / "fig_metricas_test_barras_agrupadas.png"
OUT_BARRAS_PDF = FIG_DIR / "fig_metricas_test_barras_agrupadas.pdf"
plt.savefig(OUT_BARRAS_PNG, dpi=300, bbox_inches="tight", pad_inches=0.2)
plt.savefig(OUT_BARRAS_PDF, bbox_inches="tight", pad_inches=0.2)
plt.show()
print(f"✅ Barras guardadas: {OUT_BARRAS_PNG.name}")

# --- 2. Mapa de calor ---
df_heat = df_m.set_index("clase")[metricas].copy()
df_heat.columns = labels_m

fig, ax = plt.subplots(figsize=(9, 0.7 * len(df_heat) + 2.5))
vmin = float(np.nanmin(df_heat.values)) * 0.98
im = ax.imshow(df_heat.values, cmap="Blues", vmin=max(0, vmin), vmax=1.0)
ax.set_title("Mapa de calor de métricas TEST por clase", fontsize=14, fontweight="bold")
ax.set_xticks(np.arange(len(labels_m)))
ax.set_yticks(np.arange(len(df_heat.index)))
ax.set_xticklabels(labels_m)
ax.set_yticklabels(df_heat.index)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
for i in range(df_heat.shape[0]):
    for j in range(df_heat.shape[1]):
        v = df_heat.iloc[i, j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                color="white" if v >= 0.75 else "black", fontsize=11, fontweight="bold")
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Valor de la métrica", rotation=-90, va="bottom")
ax.set_xlabel("Métrica"); ax.set_ylabel("Clase")
plt.tight_layout()

OUT_HEAT_PNG = FIG_DIR / "fig_metricas_test_mapa_calor.png"
OUT_HEAT_PDF = FIG_DIR / "fig_metricas_test_mapa_calor.pdf"
plt.savefig(OUT_HEAT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_HEAT_PDF, bbox_inches="tight")
plt.show()
print(f"✅ Mapa de calor guardado: {OUT_HEAT_PNG.name}")


## 🧮 Sección 13 – Distribución de clases por split (datos reales)

Se reconstruye la distribución leyendo directamente los `labels` del dataset
(no valores escritos a mano), enlazando el resultado del modelado con el EDA.

In [ ]:
# ============================================================
# CELDA 13 – Distribución de clases por split: barras + heatmap
# ============================================================
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = TFM_DIR / "figuras_resultados_yolo_expresiones"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def clase_dominante(lbl_path):
    clases = []
    try:
        with open(lbl_path, "r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                parts = line.strip().split()
                if len(parts) == 5:
                    cid = int(float(parts[0]))
                    if 0 <= cid < NUM_CLASSES:
                        clases.append(cid)
    except Exception:
        pass
    return max(set(clases), key=clases.count) if clases else None

# Conteo real por split y clase (1 imagen = su clase dominante)
conteo = {s: Counter() for s in SPLITS}
for split in SPLITS:
    lbl_dir = YOLO_DATASET / "labels" / split
    if not lbl_dir.exists():
        continue
    for lbl in lbl_dir.glob("*.txt"):
        c = clase_dominante(lbl)
        if c is not None:
            conteo[split][c] += 1

tabla_abs = pd.DataFrame(
    {s: [conteo[s].get(i, 0) for i in range(NUM_CLASSES)] for s in SPLITS},
    index=CLASS_NAMES,
)
print("📋 Distribución absoluta (imágenes por clase dominante):")
display(tabla_abs)

# Porcentaje dentro de cada split
tabla_pct = tabla_abs.copy().astype(float)
for s in SPLITS:
    tot = tabla_pct[s].sum()
    tabla_pct[s] = 100 * tabla_pct[s] / tot if tot > 0 else 0

# --- Barras agrupadas (absolutas) ---
x = np.arange(NUM_CLASSES)
width = 0.25
fig, ax = plt.subplots(figsize=(13, 6), constrained_layout=True)
for i, s in enumerate(SPLITS):
    vals = tabla_abs[s].values
    barras = ax.bar(x + (i - 1) * width, vals, width, label=s.upper())
    for b, v in zip(barras, vals):
        if v > 0:
            ax.text(b.get_x() + b.get_width()/2, v + max(tabla_abs.values.flatten())*0.01,
                    f"{int(v)}", ha="center", va="bottom", fontsize=8, rotation=90)
ax.set_title("Distribución absoluta de clases por split — Expresiones",
             fontsize=15, fontweight="bold")
ax.set_xlabel("Clase"); ax.set_ylabel("Número de imágenes")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=20, ha="right")
ax.legend(title="Split"); ax.grid(axis="y", alpha=0.3)
OUT_ABS = FIG_DIR / "fig_distribucion_clases_barras_absolutas.png"
plt.savefig(OUT_ABS, dpi=300, bbox_inches="tight", pad_inches=0.2)
plt.show()
print(f"✅ Guardado: {OUT_ABS.name}")

# --- Heatmap absoluto ---
fig, ax = plt.subplots(figsize=(8, 0.7 * NUM_CLASSES + 2), constrained_layout=True)
mat = tabla_abs.values
im = ax.imshow(mat, cmap="Blues")
ax.set_title("Matriz de distribución de clases por split (absoluta)",
             fontsize=14, fontweight="bold")
ax.set_xticks(np.arange(len(SPLITS))); ax.set_yticks(np.arange(NUM_CLASSES))
ax.set_xticklabels([s.upper() for s in SPLITS]); ax.set_yticklabels(CLASS_NAMES)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = int(mat[i, j])
        ax.text(j, i, f"{v}", ha="center", va="center", fontsize=11, fontweight="bold",
                color="white" if v > mat.max() * 0.55 else "black")
cbar = ax.figure.colorbar(im, ax=ax); cbar.ax.set_ylabel("Número de imágenes", rotation=-90, va="bottom")
ax.set_xlabel("Split"); ax.set_ylabel("Clase")
OUT_HEAT = FIG_DIR / "fig_matriz_distribucion_clases_absoluta.png"
plt.savefig(OUT_HEAT, dpi=300, bbox_inches="tight", pad_inches=0.2)
plt.show()
print(f"✅ Guardado: {OUT_HEAT.name}")


## 📉 Sección 14 – Curvas de entrenamiento (a partir de `results.csv`)

In [ ]:
# ============================================================
# CELDA 14 – Curvas de métricas y pérdidas del entrenamiento
# ============================================================
import matplotlib.pyplot as plt

FIG_DIR = TFM_DIR / "figuras_resultados_yolo_expresiones"
RESULTS_CSV = PROJECT_DIR / RUN_NAME / "results.csv"
print(f"📄 results.csv: {RESULTS_CSV} (existe: {RESULTS_CSV.exists()})")

if RESULTS_CSV.exists():
    dfh = pd.read_csv(RESULTS_CSV)
    dfh.columns = [c.strip() for c in dfh.columns]   # limpiar espacios
    epochs = dfh["epoch"] if "epoch" in dfh.columns else range(len(dfh))

    def col(name):
        return dfh[name] if name in dfh.columns else None

    # --- 1. Métricas por época ---
    fig, ax = plt.subplots(figsize=(11, 6))
    for c, lab in [("metrics/precision(B)", "Precision"),
                   ("metrics/recall(B)", "Recall"),
                   ("metrics/mAP50(B)", "mAP50"),
                   ("metrics/mAP50-95(B)", "mAP50-95")]:
        y = col(c)
        if y is not None:
            ax.plot(epochs, y, marker="", linewidth=2, label=lab)
    ax.set_title("Evolución de métricas durante el entrenamiento", fontsize=14, fontweight="bold")
    ax.set_xlabel("Época"); ax.set_ylabel("Valor"); ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    OUT1 = FIG_DIR / "fig_curva_metricas_entrenamiento.png"
    plt.savefig(OUT1, dpi=300, bbox_inches="tight"); plt.show()
    print(f"✅ Guardado: {OUT1.name}")

    # --- 2. Pérdidas (train vs val) ---
    fig, ax = plt.subplots(figsize=(11, 6))
    for c, lab in [("train/box_loss", "train box_loss"),
                   ("train/cls_loss", "train cls_loss"),
                   ("train/dfl_loss", "train dfl_loss"),
                   ("val/box_loss", "val box_loss"),
                   ("val/cls_loss", "val cls_loss"),
                   ("val/dfl_loss", "val dfl_loss")]:
        y = col(c)
        if y is not None:
            estilo = "--" if c.startswith("val") else "-"
            ax.plot(epochs, y, estilo, linewidth=1.8, label=lab)
    ax.set_title("Evolución de las pérdidas (train vs val)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.grid(alpha=0.3); ax.legend(ncol=2)
    plt.tight_layout()
    OUT2 = FIG_DIR / "fig_curva_perdidas_entrenamiento.png"
    plt.savefig(OUT2, dpi=300, bbox_inches="tight"); plt.show()
    print(f"✅ Guardado: {OUT2.name}")
else:
    print("⚠️ No se encontró results.csv. Ejecuta primero el entrenamiento (celda 4).")


## 🧹 Sección 15 – Liberar memoria GPU / RAM

In [ ]:
# ============================================================
# CELDA 15 – Liberar memoria GPU y RAM
# ============================================================
import gc, torch

print("=" * 70)
for v in ["model", "model_best", "results", "pred_results", "metrics_test"]:
    if v in globals():
        del globals()[v]
        print(f"🗑️  Eliminado: {v}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🔥 VRAM reservada tras limpieza: {torch.cuda.memory_reserved(0)/1024**2:.0f} MB")
print("✅ Memoria liberada.")


## 🔐 Sección 16 – Auditoría final de integridad (solo lectura)

In [ ]:
# ============================================================
# CELDA 16 – Auditoría final de integridad del proyecto
# NO entrena, NO predice, NO borra, NO mueve archivos. Solo lectura.
# ============================================================
print("=" * 70)
print("🔐 AUDITORÍA FINAL DEL PROYECTO DE EXPRESIONES")
print("=" * 70)

TEST_EVAL_DIR = PROJECT_DIR / f"{RUN_NAME}_test_eval"
FIG_DIR       = TFM_DIR / "figuras_resultados_yolo_expresiones"
PRED_DIR      = TFM_DIR / "predicciones_yolo_expresiones" / "test_visual_baseline_yolov8n"

checks = {
    "data.yaml":            DATA_YAML,
    "best.pt":              BEST_MODEL,
    "last.pt":              LAST_MODEL,
    "results.csv":          PROJECT_DIR / RUN_NAME / "results.csv",
    "carpeta eval TEST":    TEST_EVAL_DIR,
    "matriz confusión":     TEST_EVAL_DIR / "confusion_matrix.png",
    "resumen métricas CSV": TEST_EVAL_DIR / "metricas_test_resumen.csv",
    "carpeta figuras":      FIG_DIR,
    "carpeta predicciones": PRED_DIR,
}

print("\n📋 Artefactos del proyecto:")
for nombre, ruta in checks.items():
    print(f"  {'✅' if ruta.exists() else '❌'} {nombre:<22} → {ruta}")

# Conteo de imágenes por split
print("\n📊 Imágenes por split:")
total = 0
for split in SPLITS:
    d = YOLO_DATASET / "images" / split
    n = len([p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]) if d.exists() else 0
    total += n
    print(f"  {split.upper():<6}: {n:>6,}")
print(f"  {'TOTAL':<6}: {total:>6,}")

# Verificar que NO haya aumentaciones fuera de train
print("\n🧪 Aumentaciones fuera de TRAIN (no deseado):")
for split in ["val", "test"]:
    d = YOLO_DATASET / "images" / split
    aug = list(d.glob("aug_*")) if d.exists() else []
    print(f"  {split.upper()}: aug images = {len(aug):,}  {'⚠️' if aug else '✅'}")

print("\n✅ Auditoría completada (solo lectura).")


## 📝 Notas finales para la documentación

**Uso de los conjuntos.** El conjunto de *entrenamiento* ajusta los pesos del modelo;
el de *validación* monitorea el desempeño por época (Precision, Recall, mAP50, mAP50-95)
y dispara el *early stopping*; el de *prueba* se reserva para la evaluación final del
modelo ya entrenado.

**Interpretación de la matriz de confusión.** En la diagonal principal se concentran las
expresiones correctamente clasificadas. En un problema de 7 clases es esperable cierta
confusión entre expresiones visualmente cercanas (p. ej. `fear`↔`surprise` o `sad`↔`neutral`).
La fila/columna `background` corresponde a la representación interna de YOLO para falsos
positivos/negativos y **no es una octava clase** del problema.

**Continuidad.** Este notebook consume el dataset equilibrado que produjo
`TFM_EDA_YOLO_Expresiones.ipynb` y entrega: pesos (`best.pt`), métricas reales por clase,
matriz de confusión, curvas PR/F1/P/R, curvas de entrenamiento y las figuras listas para
el documento y la sustentación.
